In [8]:
# Core libraries
import pandas as pd
import numpy as np

# Train-test split
from sklearn.model_selection import train_test_split

# Model
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

# Scaling
from sklearn.preprocessing import StandardScaler

# Export Model
import joblib

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    f1_score
)

In [ ]:
# Load dataset after preprocessing (from Person 3 work)
df = pd.read_csv("..data/cleaned_data.csv")

# Quick check
df.head()

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Age_Damage_Interaction,Premium_Quintile
0,1,44,1,28.0,0,2,1,40454.0,26.0,217,1,2,3
1,1,76,1,3.0,0,1,0,33536.0,26.0,183,0,0,2
2,1,47,1,28.0,0,2,1,38294.0,26.0,27,1,2,3
3,1,21,1,11.0,1,0,0,28619.0,152.0,203,0,0,1
4,0,29,1,41.0,1,0,0,27496.0,152.0,39,0,0,1


In [10]:
# Separate input features and target variable
X = df.drop('Response', axis=1)
y = df['Response']

In [11]:
# Split dataset into train and test sets
# stratify=y ensures same class distribution in both sets (important for imbalance)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,)

In [12]:
# Check class distribution in full dataset
print("Overall distribution:")
print(y.value_counts(normalize=True) * 100)

# Check train set distribution
print("\nTrain distribution:")
print(y_train.value_counts(normalize=True) * 100)

Overall distribution:
Response
0    87.743664
1    12.256336
Name: proportion, dtype: float64

Train distribution:
Response
0    87.743807
1    12.256193
Name: proportion, dtype: float64


In [13]:
# Scaling is required for models using distance or gradients:
# Logistic Regression, KNN, SVM, ANN

num_cols = [
    'Age',
    'Annual_Premium',
    'Vintage',
    'Region_Code',
    'Policy_Sales_Channel'
]

scaler = StandardScaler()

# Fit ONLY on training data
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Transform test data
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [14]:
models = {
    "SVM": CalibratedClassifierCV(LinearSVC(class_weight='balanced', random_state=42, max_iter=10000,))
}

In [15]:
# Store trained models
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

Training SVM...


In [16]:
# Store results
results = []

for name, model in trained_models.items():
    
    # Predictions
    y_pred = model.predict(X_test)
    
    # Probabilities (for ROC-AUC)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    if y_prob is not None:
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = None
    
    print(f"\n{name}")
    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print("ROC-AUC:", auc)
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    results.append([name, acc, f1, auc])


SVM
Accuracy: 0.8769915405445119
F1 Score: 0.02023073064704899
ROC-AUC: 0.8313328742099609

Confusion Matrix:
 [[83437   163]
 [11557   121]]


In [19]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# 1. Extract the probabilities from your calibrated SVM
y_probs = model.predict_proba(X_test)[:, 1]

# 2. Test thresholds from 0.02 to 0.50 (lowered since your positive class is rare)
best_threshold = 0.5
best_f1 = 0
best_preds = None

for threshold in np.arange(0.02, 0.50, 0.01):
    preds = (y_probs >= threshold).astype(int)
    current_f1 = f1_score(y_test, preds)
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold
        best_preds = preds

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"New Best F1 Score: {best_f1:.4f}")
print(f"New Accuracy: {accuracy_score(y_test, best_preds):.4f}")
print("\nOptimized Confusion Matrix:")
print(confusion_matrix(y_test, best_preds))

Optimal Threshold: 0.20
New Best F1 Score: 0.4298
New Accuracy: 0.7420

Optimized Confusion Matrix:
[[61437 22163]
 [ 2416  9262]]


In [ ]:
# 1. Package the model and the optimal threshold together
optimized_bundle = {
    'model': model,                  # Your trained sample-weighted GradientBoosting model
    'threshold': 0.20                # Your optimal business threshold
}

# 2. Export the bundle to a single .sav file
joblib.dump(optimized_bundle, '../exports/svm_model_optimised.sav')
print("Optimized model and threshold successfully exported to 'svm_model_optimised.sav'!")


Optimized model and threshold successfully exported to 'svm_model_optimised.sav'!
